
# Train WaveNet on Realistic Multi-Pulse Traces

This example follows a production-style training workflow: a library of
slightly different asymmetric pulse shapes, variable noise and baseline drift,
Poisson-distributed pulse counts, explicit blank traces, per-trace
normalization, and validation-aware WaveNet training.

The target is the pulse-location label generated by :class:`SignalGenerator`.
For clean-trace regression, replace ``targets`` with
``dataset.clean_signals[..., None]`` and use a regression loss such as Huber.


## Imports and reproducibility



In [ ]:
import numpy as np
import tensorflow as tf

from DeepPeak.generation import DataSet, Gaussian, PoissonCount, SignalGenerator
from DeepPeak.models import TrainingConfig, WaveNet
from DeepPeak.models import plot_predictions

tf.keras.utils.set_random_seed(42)

SEQUENCE_LENGTH = 256
MAX_NUM_PEAKS = 4

## Build a Gaussian pulse kernel with acquisition variability



In [ ]:
kernel = Gaussian(
    width=(8.0, 18.0),
    position=(20.0, SEQUENCE_LENGTH - 20.0),
    amplitude=(0.5, 2.0),
)

## Generate signal and blank batches with acquisition variability



In [ ]:
generator = SignalGenerator(sequence_length=SEQUENCE_LENGTH)
acquisition = dict(
    noise_std=(0.01, 0.12),
    minimum_level=(0.02, 0.08),
    drift=(-0.003, 0.003),
    noise_profile="linear",
    noise_end_scale=(0.5, 1.0),
)

generator.add_to_set(
    n_samples=2024,
    kernel=kernel,
    peak_count=PoissonCount(bounds=(0, MAX_NUM_PEAKS), rate=(1.5, 3.5)),
    **acquisition,
)
generator.add_to_set(
    n_samples=64,
    kernel=kernel,
    peak_count=PoissonCount(bounds=(0, 0), rate=(0, 0)),
    **acquisition,
)
dataset = generator.dataset().shuffle(seed=42)

## Inspect multi-pulse examples and prepare model arrays



In [ ]:
_ = dataset.plot(
    number_of_samples=6,
    number_of_columns=3,
    randomize_signal=False,
    reference_pulse_trace=dataset.clean_signals,
)

inputs = dataset.to_model_inputs(normalization="zscore").astype(np.float32)
targets = dataset.targets().astype(np.float32)

## Train a WaveNet detector with validation callbacks



In [ ]:
wavenet = WaveNet(
    sequence_length=SEQUENCE_LENGTH,
    num_filters=32,
    num_dilation_layers=5,
    kernel_size=4,
    output_activation="sigmoid",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=("binary_accuracy",),
)

history = wavenet.fit_dataset(
    dataset,
    normalization="zscore",
    config=TrainingConfig(
        epochs=20,
        batch_size=32,
        validation_split=0.2,
        patience=5,
        monitor="val_loss",
        verbose=1,
        seed=42,
    ),
)

## Visualize training history and predictions



In [ ]:
_ = wavenet.plot_model_history(yscale="log")

plot_dataset = DataSet(
    signals=inputs[..., 0],
    labels=targets[..., 0],
    x_values=dataset.x_values,
)
figure = plot_predictions(
    wavenet,
    plot_dataset,
    n_samples=6,
    n_columns=3,
    randomize=True,
    seed=42,
)